In [1]:
for i in range(10):
    print(f' No={i}')
    

 No=0
 No=1
 No=2
 No=3
 No=4
 No=5
 No=6
 No=7
 No=8
 No=9


In [1]:
import types
class Test(object):
    def __init__(self,name):
        self.name = name
        
    def hello(self):
        print(f"Hello World {self.name}")

tx=Test("xxxxxxxxxx")
print(Test.hello)
Test.hello(tx)

t1=Test("yukihito")
print(t1.hello)
print(Test.__dict__['hello'].__get__(t1,type(t1)))

print(Test.__dict__['hello'].__get__(None,Test))
print(Test.hello)
print(Test.__dict__['hello'])

Test.hello(t1)

print("########################")
def hello2(self,name2: str):
    print(f"hello2 world | {name2:^30}|") 
    
Test.hello2=hello2
t2= Test("aaaaaa")
t2.hello2("cccccccccccccccccc")

t1.hello2 = types.MethodType(hello2,t1)
print(t1.hello2)
t1.hello2("DDDDDDDDDDDDDDDDDD")

print("%" * 30)
print(Test.hello2)


Test.hello2 = types.MethodType(hello2,Test)
Test.hello2("yukihito kun")


<function Test.hello at 0x103bb4fe0>
Hello World xxxxxxxxxx
<bound method Test.hello of <__main__.Test object at 0x1043df4d0>>
<bound method Test.hello of <__main__.Test object at 0x1043df4d0>>
<function Test.hello at 0x103bb4fe0>
<function Test.hello at 0x103bb4fe0>
<function Test.hello at 0x103bb4fe0>
Hello World yukihito
########################
hello2 world |       cccccccccccccccccc      |
<bound method hello2 of <__main__.Test object at 0x1043df4d0>>
hello2 world |       DDDDDDDDDDDDDDDDDD      |
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
<function hello2 at 0x1044ff600>
hello2 world |          yukihito kun         |


In [69]:
import types
class Test2():
    def hello3(self,name):
        print(f"hello world {name:<20}")

def hello4(self,name):
    print(f"hello4 word {name}")


Test2.hello3(None,"koji")

Test2.hello4 = hello4
print(Test2.__dict__)
print(type(Test2.__dict__))

Test2.hello4(Test2,"yukihito")
Test2.hello4 = types.MethodType(hello4,Test2)
Test2.hello4("yukihito bouzu")
        
class Descobject():
    def __init__(self,func):
        self.func = func
    def __get__(self,instance,owner):
        def method(*args,**kwargs):
            return self.func(*args,**kwargs)        
        return method

@Descobject
def hello5(name):
    print(f"hello5 world {name}")

class Test6():
    pass

Test6.hello5=hello5
Test6.hello5("デスクリプタの理解")


hello world koji                
{'__module__': '__main__', 'hello3': <function Test2.hello3 at 0x10f6920c0>, '__dict__': <attribute '__dict__' of 'Test2' objects>, '__weakref__': <attribute '__weakref__' of 'Test2' objects>, '__doc__': None, 'hello4': <function hello4 at 0x10f691940>}
<class 'mappingproxy'>
hello4 word yukihito
hello4 word yukihito bouzu
hello5 world デスクリプタの理解


In [1]:
import logging

logging.basicConfig(level=logging.INFO)

class LoggedAgeAccess:

    def __get__(self, obj, objtype=None):
        value = obj._age
        logging.info('Accessing %r giving %r', 'age', value)
        return value

    def __set__(self, obj, value):
        logging.info('Updating %r to %r', 'age', value)
        obj._age = value

class Person:

    age = LoggedAgeAccess()             # Descriptor instance

    def __init__(self, name, age):
        self.name = name                # Regular instance attribute
        self.age = age                  # Calls __set__()

    def birthday(self):
        self.age += 1                   # Calls both __get__() and __set__()



mary = Person('Mary M', 30) 
dave = Person('David D', 40)
vars(mary) 
vars(dave)
mary.age
mary.birthday() 
dave.name 
dave.age

INFO:root:Updating 'age' to 30
INFO:root:Updating 'age' to 40
INFO:root:Accessing 'age' giving 30
INFO:root:Accessing 'age' giving 30
INFO:root:Updating 'age' to 31
INFO:root:Accessing 'age' giving 40


40

In [2]:
import logging

logging.basicConfig(level=logging.INFO)

class LoggedAccess:

    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = '_' + name

    def __get__(self, obj, objtype=None):
        value = getattr(obj, self.private_name)
        logging.info('Accessing %r giving %r', self.public_name, value)
        return value

    def __set__(self, obj, value):
        logging.info('Updating %r to %r', self.public_name, value)
        setattr(obj, self.private_name, value)

class Person:

    name = LoggedAccess()                # First descriptor instance
    age = LoggedAccess()                 # Second descriptor instance

    def __init__(self, name, age):
        self.name = name                 # Calls the first descriptor
        self.age = age                   # Calls the second descriptor

    def birthday(self):
        self.age += 1

vars(vars(Person)['name'])
vars(vars(Person)['age'])

pete = Person('Peter P', 10)
kate = Person('Catherine C', 20)

vars(pete)
vars(kate)


INFO:root:Updating 'name' to 'Peter P'
INFO:root:Updating 'age' to 10
INFO:root:Updating 'name' to 'Catherine C'
INFO:root:Updating 'age' to 20


{'_name': 'Catherine C', '_age': 20}

In [18]:
class Descriptor:
    def __init__(self, value=None):
        self.value = value

    def __get__(self, instance, owner):
        return self.value

    def __set__(self, instance, value):
        self.value = value

    def __set_name__(self, owner, name):
        self.owner = owner
        self.name = name
        print(f'Descriptor added to {owner.__name__} with name {name}')

class MyClass:
    attr = Descriptor()

# クラスが作成されるときに、__set_name__が呼び出される
obj = MyClass()


Descriptor added to MyClass with name attr


In [1]:
class MultiDelegate:
    def __init__(self, *delegatees):
        self.delegatees = delegatees

    def __getattr__(self, name):
        for delegatee in self.delegatees:
            if hasattr(delegatee, name):
                return getattr(delegatee, name)
        raise AttributeError(f"'{self.__class__.__name__}' object has no attribute '{name}'")

    def __setattr__(self, name, value):
        if name == 'delegatees':
            super().__setattr__(name, value)
        else:
            for delegatee in self.delegatees:
                if hasattr(delegatee, name):
                    setattr(delegatee, name, value)
                    return
            raise AttributeError(f"'{self.__class__.__name__}' object has no attribute '{name}'")

# Delegatee classes
class Delegatee1:
    def method1(self):
        return "Delegatee1: method1 called"

class Delegatee2:
    def method2(self):
        return "Delegatee2: method2 called"

# Delegator class
class Delegator(MultiDelegate):
    def own_method(self):
        return "Delegator's own method called"

# Usage example
delegatee1 = Delegatee1()
delegatee2 = Delegatee2()
delegator = Delegator(delegatee1, delegatee2)

print(delegator.own_method())  # Delegator's own method called
print(delegator.method1())  # Delegatee1: method1 called
print(delegator.method2())  # Delegatee2: method2 called


Delegator's own method called
Delegatee1: method1 called
Delegatee2: method2 called


In [2]:
class Element:
    def __init__(self, name):
        self.name = name
        self.children = []
        self.attributes = {}

    def add_child(self, child):
        self.children.append(child)
        return self

    def set_attr(self, key, value):
        self.attributes[key] = value
        return self

    def __str__(self):
        attrs = ' '.join(f'{key}="{value}"' for key, value in self.attributes.items())
        if attrs:
            attrs = ' ' + attrs
        inner_html = ''.join(str(child) for child in self.children)
        return f'<{self.name}{attrs}>{inner_html}</{self.name}>'

class Delegate:
    def __init__(self, delegatee):
        self.delegatee = delegatee

    def __getattr__(self, name):
        return getattr(self.delegatee, name)

    def __setattr__(self, name, value):
        if name == 'delegatee':
            super().__setattr__(name, value)
        else:
            setattr(self.delegatee, name, value)

class HtmlElement(Element):
    pass

class BodyElement(Element):
    pass

class PElement(Element):
    pass

class HtmlDSL(Delegate):
    def __init__(self):
        self.html = HtmlElement('html')
        super().__init__(self.html)

    def body(self):
        body_element = BodyElement('body')
        self.html.add_child(body_element)
        self.delegatee = body_element
        return self

    def p(self, text):
        p_element = PElement('p')
        p_element.add_child(TextElement(text))
        self.delegatee.add_child(p_element)
        return self

class TextElement(Element):
    def __init__(self, text):
        self.text = text

    def __str__(self):
        return self.text

# 使用例
dsl = HtmlDSL()
dsl.body().p("Hello, World!").p("This is a paragraph.")
print(dsl)


RecursionError: maximum recursion depth exceeded

In [3]:
class Node:
    def __init__(self, name):
        self.name = name
        self.children = []

    def add(self, child):
        self.children.append(child)

    def __repr__(self):
        return f"Node({self.name})"

class Delegate:
    def __init__(self, delegatee):
        self.__dict__['delegatee'] = delegatee

    def __getattr__(self, name):
        delegatee = self.__dict__['delegatee']
        return getattr(delegatee, name)

    def __setattr__(self, name, value):
        if name == 'delegatee':
            self.__dict__[name] = value
        else:
            delegatee = self.__dict__['delegatee']
            setattr(delegatee, name, value)

class DSLContext(Delegate):
    def __init__(self, node):
        super().__init__(node)

    def node(self, name):
        new_node = Node(name)
        self.delegatee.add(new_node)
        return DSLContext(new_node)

# 使用例
root = Node('root')
dsl = DSLContext(root)

dsl.node('child1').node('grandchild1')
dsl.node('child2')

print(root.children)
for child in root.children:
    print(child, '->', child.children)



[Node(child1), Node(child2)]
Node(child1) -> [Node(grandchild1)]
Node(child2) -> []


In [15]:
class Test():
    def __init__(self,name):
        self.name = name
        
    def __str__(self):
        return f"hello {self.name}"

    def __repr__(self):
        return f"Class Name = Test"

t1 = Test("koji")
print(t1)
print(repr(t1))


hello koji
Class Name = Test


a=10000000
print(f'{a:,}')

In [2]:
a=10000000 
print(f'{a:,}')
print(f'{a:_}')

10,000,000
10_000_000


In [4]:
passwd = 'password'

print(f'{passwd:*^20}')
# ******password******

print(f'{passwd:*<20}')
# password************

print(f'{passwd:ば>20}')
# ばばばばばばばばばばばばpassword
print(f'{passwd:0>20}')

******password******
password************
ばばばばばばばばばばばばpassword
000000000000password


In [5]:
n = [
    ('a', 1),
    ('b', 2),
    ('c', 3),
]
for i, (item, count) in enumerate(n):
    print(f'{i}: '
          f'{item:<10} = '
          f'{count}')


0: a          = 1
1: b          = 2
2: c          = 3


In [14]:
'''\
 aaaaaaaaaaaaaaaa
 bbbbbbbbbbbbbbbb
 あああああああああああ
'''



' aaaaaaaaaaaaaaaa\n bbbbbbbbbbbbbbbb\n あああああああああああ\n'

In [20]:
class testobj(object):
    def __init__(self,name):
        self.name = name
        
    def getname(self):
        print(f'hello {self.name}')
        return testobj("koko")
        

t1 = testobj("koji")
t1.getname().getname()

hello koji
hello koko


In [21]:
class Delegate(object):
    def __init__(self,delegatee):
        self.delegatee = delegatee

    def __getattr__(self,name):
        print(f'呼び出されたメソッド {name=}')
        return getattr(self.delegatee,name)

class Delegatee():
    def __init__(self,name):
        self.name = name
    
    def hello1(self):
        print(f"hello world1 name= {self.name}")
        return self

    def hello2(self):
        print(f"hello world2 {self.name=}")
        return self
        
class Test(Delegate):
    def __init__(self,delegatee):
        self.delegatee = delegatee
        super().__init__(delegatee)
    
    def hellox(self,name):
        print(f'Hello {name}')

d1 = Delegatee("yukihito2")
t1 = Test(d1)
t1.hellox("koji")
t1.hello1().hello2().hello1()
t1.hello2()

Hello koji
呼び出されたメソッド name='hello1'
hello world1 name= yukihito2
hello world2 self.name='yukihito2'
hello world1 name= yukihito2
呼び出されたメソッド name='hello2'
hello world2 self.name='yukihito2'


In [4]:
class Test2():
    def __init__(self):
        self.__prt_()
        
    def __prt_(self):
        print("hello world")
t1= Test2()
t1._Test2__prt_()

hello world
hello world


class Base:
    def base_method(self):
        print(f"Called from: {self.__class__.__name__}")
        if hasattr(self, 'derived_attr'):
            print(f"Derived attribute: {self.derived_attr}")

class Derived(Base):
    def __init__(self):
        self.derived_attr = "I am from Derived class"

# Derivedクラスをインスタンス化
derived_instance = Derived()

# DerivedクラスのインスタンスからBaseクラスのメソッドを呼び出す
derived_instance.base_method()


In [20]:
def calc(y,z):
    def calc2(x):
        return x + y + z
    return calc2

f1=calc(10,5)
f1(6)

21